In [1]:
from sklearn.externals.array_api_compat import torch
%load_ext autoreload
%autoreload 2

# Finding Better Prompts from Retrieved Examples

## Goal

This notebook runs the complete BSc experiment:

```text
Bias-in-Bios → hold out one column → retrieve examples → build prompts
→ score every allowed label as a continuation of each prompt
→ select one prompt per language model on validation rows
→ evaluate each selected prompt once on separate test rows
→ inspect predictive quality and group disparities
```

`hard_text` is always input. With target **profession**, gender is the other input and audit group. With target **gender**, profession is the other input and audit group. The structured target value is never included in a validation or test prompt.

Every candidate label is scored against the same master prompt, examples, query, and chat wrapper.

## Setup and assumptions

Use this DataSpell interpreter:

```text
/opt/homebrew/anaconda3/envs/prompt-selection/bin/python
```

Install `requirements.txt` once. Edit `config.yaml` before running if you want a different target, data size, retrieval method, number/order of examples, master prompt, model, or validation ranking objective.

Language-model inference uses Hugging Face Transformers directly. Models are loaded and released sequentially. Retrieval encoders use SentenceTransformers.

The first real run can download model files and builds one persistent, model-named LanceDB table per embedding model. Matching later runs reuse those tables instead of embedding the training pool again. Delete `data/lancedb/` before changing data or embedding settings. Semantic conditions compare Qwen3-Embedding-8B and BAAI/bge-large-en-v1.5; training documents remain raw `hard_text`, while each encoder receives its own query prefix.

The `label_score` policy computes a mean conditional token log-score for every allowed label and chooses the largest. Scores are relative rankings rather than calibrated probabilities, and inference errors stop the run.

In [2]:
from datasets import load_dataset
from tqdm import tqdm

raw_train = load_dataset(
    "LabHC/bias_in_bios",
    split="train",
)

print(len(raw_train))  # 257478
print(raw_train[0])

257478
{'hard_text': 'He is also the project lead of and major contributor to the open source assembler/simulator "EASy68K." He earned a master’s degree in computer science from the University of Michigan-Dearborn, where he is also an adjunct instructor. Downloads/Updates', 'profession': 21, 'gender': 0}


In [3]:
from pipeline import load_llm, clear_model_memory
import inspect

# a = ['Qwen/Qwen3.6-27B', 'Qwen/Qwen3.5-27B', 'Qwen/Qwen3.6-35B-A3B', 'google/gemma-4-31B-it']
# a = ['google/gemma-4-31B-it']
# b = ['6a9e13bd6fc8f0983b9b99948120bc37f49c13e9', 'fc05daec18b0a78c049392ed2e771dde82bdf654',
#      '995ad96eacd98c81ed38be0c5b274b04031597b0', '842da3794eaa0b77d5f08bae87a17459d91ff475']
# b = ['842da3794eaa0b77d5f08bae87a17459d91ff475']
# c = 'bfloat16'
#
# for x, y in zip(a, b):
#     tokenizer, model = load_llm(x, y, 'mps', c)
#     forward_parameters = inspect.signature(model.forward).parameters
#     print(model.name_or_path, 'logits_to_keep' in forward_parameters)
#
#     del model, tokenizer
#     clear_model_memory('mps')

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

google/gemma-4-31B-it True


In [4]:
tokenizer, model = load_llm('Qwen/Qwen3.6-27B', '6a9e13bd6fc8f0983b9b99948120bc37f49c13e9', 'mps', c)

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

In [25]:
{'hi': 1, 'how': 2}.__getitem__

<function dict.__getitem__(key, /)>

In [10]:
from pathlib import Path
import sys

import yaml
from IPython.display import Image, display

# DataSpell can start in this folder or in its parent project folder.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").exists():
    candidate = PROJECT_ROOT / "retrieval-guided-master-prompt-selection"
    if (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
    else:
        raise FileNotFoundError("Could not find config.yaml")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline import (  # noqa: E402 - imported after locating the project
    load_config,
    load_data,
    render_input,
    run_experiment,
    task_settings,
    validate_config,
)

CONFIG_PATH = PROJECT_ROOT / "config.yaml"
config = load_config(CONFIG_PATH)
validate_config(config)
DISPLAY_LIMIT = 40
target, audit_column, professions, target_labels = task_settings(config)

language_models = config["model"]["language_models"]
model_count = len(language_models)
per_model_condition_count = (
        len(config["retrieval"]["methods"])
        * len(config["retrieval"]["embedding_models"])
        * len(config["retrieval"]["k_values"])
        * len(config["retrieval"]["example_orders"])
        * len(config["prompt_templates"])
)
condition_count = model_count * per_model_condition_count
validation_row_count = (
        len(professions)
        * 2
        * int(config["dataset"]["validation_per_profession_gender"])
)
test_row_count = (
        len(professions)
        * 2
        * int(config["dataset"]["test_per_profession_gender"])
)

print(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))
print(f"Held-out target: {target}")
print(f"Visible input columns: hard_text + {audit_column}")
print(f"Allowed answers: {target_labels}")
print("Configured language models:")
for model_entry in language_models:
    print(f"- {model_entry['id']}")
print(f"Validation conditions per model: {per_model_condition_count}")
print(f"Total validation conditions: {condition_count}")
print(f"Validation rows per condition: {validation_row_count}")
print(f"Final-test rows per selected model condition: {test_row_count}")
print(
    "Row-condition evaluations:",
    condition_count * validation_row_count + model_count * test_row_count,
)
print("Each allowed label is scored directly with the configured Hugging Face model.")

defaults:
  target: profession
  seed: 42
  output_dir: results
  ranking_metric: macro_f1
  ranking_direction: maximize
dataset:
  file: data/bias_in_bios.jsonl
  hub_id: LabHC/bias_in_bios
  professions: all
  shuffle_seed: 42
  train_size: all
  validation_per_profession_gender: 5
  test_per_profession_gender: 10
retrieval:
  methods:
  - semantic
  - balanced_semantic
  k_values:
  - 2
  - 4
  example_orders:
  - as_retrieved
  embedding_models:
  - id: Qwen/Qwen3-Embedding-8B
    dimension: 4096
    max_sequence_length: 1024
    batch_size: 16
    dtype: bfloat16
    query_prompt: 'Instruct: Given a biography, retrieve biographies with similar
      semantic content.

      Query:'
  - id: BAAI/bge-large-en-v1.5
    dimension: 1024
    max_sequence_length: 512
    batch_size: 64
    dtype: float32
    query_prompt: 'Represent this sentence for searching relevant passages: '
  lancedb_path: data/lancedb
prompt_templates:
  neutral: Predict the held-out {target} from the supplied fi

## Step 1 — Check the three data partitions and visible fields

Training rows form the retrieval pool. Balanced validation cells select the prompt; disjoint balanced test cells estimate its final performance. The next cell loads the data cache—or downloads missing rows—and shows exactly what may enter a query prompt; it does not load either model.

In [ ]:
train_rows, validation_rows, test_rows_data, dataset_counts = load_data(config, PROJECT_ROOT)

dataset_counts

In [ ]:
first_rendered_query = render_input(validation_rows[0], target)
print("First model-visible validation query:")
print(first_rendered_query)
print(
    f"True {target} is stored separately for evaluation:",
    validation_rows[0][target],
)

## Step 2 — Select within each model, then evaluate once on test

Every prompt condition receives the same validation rows. The configured metric ranks conditions separately within each language model. Exactly one winner per model is then inferred on untouched test rows, avoiding selection on the final evaluation set and keeping every model's search budget equal.

In [ ]:
run = run_experiment(config, PROJECT_ROOT, progress=print)
best_prompts_path = run["best_prompts"]
print("Results folder:", run["run_dir"])
print("Selected prompts file:", best_prompts_path)

configured_model_ids = {entry["id"] for entry in language_models}
selected_validation = run["validation_results"].loc[
    run["validation_results"]["selected_for_test"].astype(bool)
]
assert len(selected_validation) == model_count
assert set(selected_validation["model"]) == configured_model_ids
assert len(run["results"]) == model_count
assert set(run["results"]["model"]) == configured_model_ids
assert set(run["predictions"]["predicted_label"]).issubset(target_labels)
print("Selection check passed: one validation winner and one final-test result per model.")
print("Closed-set check passed: every prediction is an allowed label.")

## Step 3 — Compare validation conditions and read the final results

`validation_results` ranks conditions within each model and marks one winner per model. `results` contains one independent final-test row for every selected model condition. The plot compares validation quality and disparities and shows the corresponding final-test scores.

In [ ]:
ranking_metric = config["defaults"]["ranking_metric"]
summary_columns = [
    "model", "rank", "selected_for_test", "condition",
    "accuracy", "macro_f1", "balanced_accuracy",
    "matthews_correlation_coefficient", "cohen_kappa",
    "worst_group_accuracy", "group_accuracy_difference",
    "max_demographic_parity_difference",
    "max_equal_opportunity_difference",
    "max_equalized_odds_difference",
]
if ranking_metric not in summary_columns:
    summary_columns.append(ranking_metric)
summary_columns = [
    column for column in summary_columns if column in run["validation_results"].columns
]
print("Validation prompt ranking (rank resets within each model):")
display(
    run["validation_results"]
    .sort_values(["model", "rank"], kind="stable")[summary_columns]
)

final_columns = [
    "model", "condition", "accuracy", "macro_f1", "balanced_accuracy",
    "matthews_correlation_coefficient", "cohen_kappa",
    "worst_group_accuracy", "group_accuracy_difference",
    "max_demographic_parity_difference",
    "max_equal_opportunity_difference",
    "max_equalized_odds_difference",
]
if ranking_metric not in final_columns:
    final_columns.append(ranking_metric)
final_columns = [column for column in final_columns if column in run["results"].columns]
print("Independent final-test result for each model:")
display(run["results"].sort_values("model", kind="stable")[final_columns])

display(Image(filename=str(run["plot"])))

## Step 4 — Inspect final-test denominators and failure modes

The summary is not enough by itself. Per-class scores show which labels fail, and group rates show the supports behind each disparity. The next cell displays final-test details for every selected model condition; all validation details are in the same returned tables with `evaluation_split == "validation"`.

In [ ]:
test_detail_tables = {
    "Per-class metrics": run["class_metrics"].query("evaluation_split == 'test'"),
    "Group disparities": run["fairness_metrics"].query("evaluation_split == 'test'"),
    "Group rates": run["group_metrics"].query("evaluation_split == 'test'"),
    "Confusion counts": run["confusion_matrix"].query("evaluation_split == 'test'"),
}
for table_name, table in test_detail_tables.items():
    preview = table.groupby("model", sort=False, group_keys=False).head(DISPLAY_LIMIT)
    print(
        f"{table_name}: showing {len(preview)} of {len(table)} rows "
        f"(up to {DISPLAY_LIMIT} per model)"
    )
    display(preview)

## Metric and label-scoring guide

For class $c$:

$$Precision_c=\frac{TP_c}{TP_c+FP_c},\quad
Recall_c=\frac{TP_c}{TP_c+FN_c},\quad
F1_c=\frac{2TP_c}{2TP_c+FP_c+FN_c}$$

Accuracy is the fraction of all correct rows. Macro averages give every target class equal weight; weighted averages use class support; balanced accuracy is macro recall. MCC and Cohen's $\kappa$ summarize agreement while accounting for more of the confusion structure.

For `label_score`, allowed class $c$ has continuation tokens $T_c$:

$$score(c)=\frac{1}{|T_c|}\sum_j
\log P(t_j\mid prompt,t_{<j}),\qquad
\hat c=\arg\max_{c\in labels}score(c)$$

Mean normalization reduces the automatic disadvantage of multi-token labels. These are relative ranking scores, not calibrated probabilities. Closed-set choice guarantees a configured label; it does not guarantee correctness or fairness.

For audit group $g$:

$$SR_{c,g}=P(\hat Y=c\mid A=g),\quad
TPR_{c,g}=P(\hat Y=c\mid Y=c,A=g),\quad
FPR_{c,g}=P(\hat Y=c\mid Y\ne c,A=g)$$

The main symmetric disparities are the range across groups: demographic-parity difference uses selection rate, equal-opportunity difference uses TPR, and equalized-odds difference is the larger of the TPR and FPR ranges. Difference metrics are better near 0; demographic-parity ratio is better near 1.

See `README.md` for every formula, undefined-case rule, configuration effect, and interpretation caveat.

## Step 5 — Inspect selected prompts and their test predictions

`best_prompts.txt` contains one resolved validation-selected master instruction per model, with hyperparameters, validation score, and final-test score. Full prompts vary by query because their retrieved examples differ, so they remain in `predictions.csv`.

`label_scores` is a JSON mapping from each allowed label to its mean conditional token log-score.

In [ ]:
print(Path(best_prompts_path).read_text(encoding="utf-8"))
selected_conditions = set(selected_validation["condition"])
prediction_columns = [
    "evaluation_split", "query_id", "target", "true_label", "audit_group",
    "predicted_label", "model", "condition", "retrieval",
    "embedding_model", "k", "example_order", "prompt_name", "label_scores",
]
prediction_columns = [
    column for column in prediction_columns if column in run["predictions"].columns
]
selected_test_predictions = run["predictions"].loc[
    run["predictions"]["evaluation_split"].eq("test")
    & run["predictions"]["condition"].isin(selected_conditions)
    ]
selected_test_preview = (
    selected_test_predictions
    .groupby("model", sort=False, group_keys=False)
    .head(DISPLAY_LIMIT)
)
print(
    f"Selected test predictions: showing {len(selected_test_preview)} of "
    f"{len(selected_test_predictions)} rows "
    f"(up to {DISPLAY_LIMIT} per model across {model_count} models)"
)
display(selected_test_preview[prediction_columns])

## Next run

Change `defaults.target` from `profession` to `gender` and run all cells again. The same prompt candidates are searched, but each target and model receives its own validation winner. Do not directly compare the two task scores as though they had identical meanings: they have different class sets, and profession is an audit subgroup—not a protected attribute—when gender is the target.

Use the configured validation and test cell sizes for thesis runs. Closed-set label scoring is deterministic, so changing inference seeds does not create independent model evidence.

The optional Gradio UI exposes the same YAML, guidance, tables, and plot and calls this exact pipeline. Run `app.py` in DataSpell when needed.